# 🏦 SGCI — Framework DQ Simplifié
### Contrôles DQ + Export CSV si toutes les dimensions sont OK
---

**Logique du pipeline :**

```
Extraction Teradata
       ↓
   run_dq()  →  5 contrôles
       ↓
  Toutes OK ?
  ┌────┴────┐
 OUI       NON
  ↓         ↓
Export    Pas d'export
CSV       (erreur loggée)
  ↓         ↓
 log()    log()
```

| Contrôle | Question |
|---|---|
| Unicité | Y a-t-il des lignes entièrement dupliquées ? |
| Complétude | Le nombre de lignes est-il dans l'intervalle attendu ? |
| Fraîcheur | Les données ont-elles des lignes dans la période d'extraction ? |
| Cohérence | Aucune colonne n'est-elle entièrement vide ? |
| Validité | Les colonnes sont-elles dans le bon type ? |


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import logging
import os
import time
import warnings
from datetime import datetime, date, timedelta
from pathlib import Path
import pytz
warnings.filterwarnings('ignore')

LOG_DIR       = "./JOURNAL/"
EXPORT_DIR    = "./EXPORTS/"
LOG_FILE_NAME = "journal_dq.csv"
TZ_ABIDJAN    = pytz.timezone("Africa/Abidjan")

Path(LOG_DIR).mkdir(parents=True, exist_ok=True)
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)

current_directory = os.getcwd()
USER = current_directory.split("/")[2] if len(current_directory.split("/")) > 2 else "sgci_user"

def setup_logger(project_name, log_dir):
    Path(log_dir).mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(project_name)
    logger.setLevel(logging.DEBUG)
    if logger.handlers:
        logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
                             datefmt="%Y-%m-%d %H:%M:%S")
    fh = logging.FileHandler(f"{log_dir}/{project_name}.log", encoding="utf-8")
    fh.setLevel(logging.DEBUG); fh.setFormatter(fmt); logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO); ch.setFormatter(fmt); logger.addHandler(ch)
    logger.propagate = False
    return logger

logger = setup_logger("sgci_dq", LOG_DIR)
print(f"✅ Setup OK")
print(f"   Logs    → {LOG_DIR}")
print(f"   Exports → {EXPORT_DIR}")


## 1. Connexion Teradata

In [ ]:
def get_connection():
    """Connexion sécurisée via variables d'environnement."""
    import teradatasql
    return teradatasql.connect(
        host     = os.getenv("TD_HOST"),
        user     = os.getenv("TD_USER"),
        password = os.getenv("TD_PASSWORD"),
        logmech  = "LDAP"
    )

def extraire(query: str, nom: str) -> pd.DataFrame:
    """Exécute une requête Teradata et retourne un DataFrame."""
    logger.info(f"[{nom}] Extraction en cours...")
    t0 = time.time()
    with get_connection() as con:
        df = pd.read_sql(query, con)
    logger.info(f"[{nom}] {len(df):,} lignes extraites en {round(time.time()-t0,2)}s")
    return df

print("✅ Connexion définie — décommenter extraire() en production")


## 2. Les 5 contrôles DQ

In [ ]:
# ── Unicité : lignes entièrement dupliquées ───────────────────────────────────
def check_unicite(df: pd.DataFrame) -> dict:
    n = df.duplicated(keep=False).sum()
    if n > 0:
        return {"statut": "ECHEC", "message": f"{n} ligne(s) entièrement dupliquée(s)"}
    return {"statut": "OK", "message": "Aucun doublon"}


# ── Complétude : nombre de lignes dans [min, max] ─────────────────────────────
def check_completude(df: pd.DataFrame, min_lignes: int, max_lignes: int) -> dict:
    n = len(df)
    if n < min_lignes:
        return {"statut": "ECHEC",
                "message": f"{n} ligne(s) — en dessous du minimum ({min_lignes})"}
    if n > max_lignes:
        return {"statut": "ECHEC",
                "message": f"{n} ligne(s) — au-dessus du maximum ({max_lignes})"}
    return {"statut": "OK", "message": f"{n} ligne(s) dans [{min_lignes}, {max_lignes}]"}


# ── Fraîcheur : lignes dans la période d'extraction ──────────────────────────
def check_fraicheur(df: pd.DataFrame, col_date: str,
                    date_debut: date, date_fin: date) -> dict:
    if col_date not in df.columns:
        return {"statut": "ECHEC", "message": f"Colonne '{col_date}' absente"}
    dates = pd.to_datetime(df[col_date], errors="coerce").dt.date
    n = dates.between(date_debut, date_fin).sum()
    if n == 0:
        return {"statut": "ECHEC",
                "message": (f"Aucune ligne dans [{date_debut}, {date_fin}] — "
                            f"dates trouvées : {dates.min()} → {dates.max()}")}
    return {"statut": "OK", "message": f"{n} ligne(s) dans [{date_debut}, {date_fin}]"}


# ── Cohérence : aucune colonne entièrement vide ───────────────────────────────
def check_coherence(df: pd.DataFrame) -> dict:
    vides = [col for col in df.columns if df[col].isnull().all()]
    if vides:
        return {"statut": "ECHEC", "message": f"Colonne(s) entièrement vide(s) : {vides}"}
    return {"statut": "OK", "message": "Aucune colonne entièrement vide"}


# ── Validité : types des colonnes ─────────────────────────────────────────────
def check_validite(df: pd.DataFrame, types_attendus: dict) -> dict:
    erreurs = []
    for col, type_attendu in types_attendus.items():
        if col not in df.columns:
            erreurs.append(f"'{col}' absente"); continue
        if df[col].dropna().empty:
            continue
        if type_attendu in (int, float):
            if not pd.api.types.is_numeric_dtype(df[col]):
                erreurs.append(f"'{col}' : attendu numérique, trouvé {df[col].dtype}")
        elif type_attendu == str:
            if not (pd.api.types.is_string_dtype(df[col]) or
                    pd.api.types.is_object_dtype(df[col])):
                erreurs.append(f"'{col}' : attendu texte, trouvé {df[col].dtype}")
        elif type_attendu == bool:
            if not pd.api.types.is_bool_dtype(df[col]):
                erreurs.append(f"'{col}' : attendu booléen, trouvé {df[col].dtype}")
    if erreurs:
        return {"statut": "ECHEC", "message": " | ".join(erreurs)}
    return {"statut": "OK", "message": "Tous les types sont conformes"}


print("✅ Les 5 contrôles définis")


## 3. Pipeline principal — run_dq()

> Si **toutes** les dimensions sont OK → le CSV est exporté dans `./EXPORTS/`.
> Si au moins une dimension est en ECHEC → pas d'export, erreur loggée.


In [ ]:
def run_dq(df: pd.DataFrame,
            nom: str,
            min_lignes: int,
            max_lignes: int,
            col_date: str,
            date_debut: date,
            date_fin: date,
            types_attendus: dict) -> dict:
    """
    Lance les 5 contrôles DQ.
    Si toutes les dimensions sont OK → exporte les données en CSV.

    Args:
        df             : DataFrame extrait de Teradata
        nom            : Nom de l'extraction (utilisé pour nommer le fichier CSV)
        min_lignes     : Nombre minimum de lignes attendu
        max_lignes     : Nombre maximum de lignes attendu
        col_date       : Colonne de date à vérifier pour la fraîcheur
        date_debut     : Début de la période d'extraction attendue
        date_fin       : Fin de la période d'extraction attendue
        types_attendus : {"colonne": type} — ex: {"AMOUNT": float, "TRN_REF": str}

    Returns:
        dict avec statut global, détail par contrôle, et chemin du CSV si exporté
    """
    t0 = time.time()
    logger.info("=" * 50)
    logger.info(f"DQ — {nom.upper()} | {len(df):,} lignes | {df.shape[1]} colonnes")
    logger.info("=" * 50)

    # ── 5 contrôles ───────────────────────────────────────────────────────────
    controles = {
        "Unicité"    : check_unicite(df),
        "Complétude" : check_completude(df, min_lignes, max_lignes),
        "Fraîcheur"  : check_fraicheur(df, col_date, date_debut, date_fin),
        "Cohérence"  : check_coherence(df),
        "Validité"   : check_validite(df, types_attendus),
    }

    # ── Affichage ─────────────────────────────────────────────────────────────
    all_ok = all(r["statut"] == "OK" for r in controles.values())

    print(f"\n{'─'*55}")
    print(f"  DQ — {nom}")
    print(f"{'─'*55}")
    for nom_ctrl, res in controles.items():
        icone = "✅" if res["statut"] == "OK" else "❌"
        print(f"  {icone} {nom_ctrl:<14} {res['message']}")
        if res["statut"] == "OK":
            logger.info(f"[{nom_ctrl}] OK — {res['message']}")
        else:
            logger.error(f"[{nom_ctrl}] ECHEC — {res['message']}")

    duree      = round(time.time() - t0, 2)
    export_csv = None

    # ── Export CSV si toutes les dimensions sont OK ───────────────────────────
    print(f"{'─'*55}")
    if all_ok:
        ts         = datetime.now().strftime("%Y%m%d_%H%M%S")
        nom_clean  = nom.lower().replace(" ", "_")
        export_csv = os.path.join(EXPORT_DIR, f"{nom_clean}_{ts}.csv")
        df.to_csv(export_csv, sep=";", index=False, encoding="utf-8")
        print(f"  📁 Export CSV : {export_csv}")
        logger.info(f"Export CSV → {export_csv} ({len(df):,} lignes)")
    else:
        nb_echecs = sum(1 for r in controles.values() if r["statut"] == "ECHEC")
        print(f"  ⛔ Pas d'export — {nb_echecs} dimension(s) en ECHEC")
        logger.warning(f"Export annulé — {nb_echecs} dimension(s) en ECHEC")

    verdict = "✅ VALIDE" if all_ok else "❌ NON VALIDE"
    print(f"  Résultat : {verdict} | Durée : {duree}s")
    print(f"{'─'*55}\n")

    return {
        "nom"        : nom,
        "timestamp"  : datetime.now().isoformat(),
        "nb_lignes"  : len(df),
        "valide"     : all_ok,
        "controles"  : controles,
        "duree_s"    : duree,
        "export_csv" : export_csv   # None si non exporté
    }


print("✅ run_dq() défini — Export CSV automatique si toutes les dimensions sont OK")


## 4. Journalisation — log()

In [ ]:
def log(nom: str, rapport: dict, frequence: str,
        date_debut: date, date_fin: date) -> None:
    """Enregistre le résultat dans le journal CSV cumulatif."""
    detail_erreurs = " | ".join([
        f"{k}: {v['message']}"
        for k, v in rapport["controles"].items()
        if v["statut"] == "ECHEC"
    ])

    record = {
        "Execution_Date" : rapport["timestamp"],
        "User"           : USER,
        "Extraction"     : nom,
        "Frequency"      : frequence,
        "Data_Begin"     : str(date_debut),
        "Data_End"       : str(date_fin),
        "Row_Number"     : rapport["nb_lignes"],
        "Duration(s)"    : rapport["duree_s"],
        "Status"         : "Good" if rapport["valide"] else "Bad",
        "Export_CSV"     : rapport["export_csv"] or "",
        "Error_Reason"   : detail_erreurs
    }

    df_new    = pd.DataFrame([record])
    full_path = os.path.join(LOG_DIR, LOG_FILE_NAME)
    if os.path.isfile(full_path):
        df_old = pd.read_csv(full_path, sep=";", encoding="utf-8")
        df_new = pd.concat([df_old, df_new], ignore_index=True)
    df_new.to_csv(full_path, sep=";", index=False, encoding="utf-8")

print("✅ log() défini — colonne Export_CSV ajoutée au journal")


## 5. Données de démonstration

In [ ]:
hier = datetime.now().date() - timedelta(days=1)
np.random.seed(42)

# ── Extraction 1 : Transactions — DQ OK → export attendu ─────────────────────
n1 = 250
df_transactions = pd.DataFrame({
    "TRN_REF"          : [f"TRN{str(i).zfill(8)}" for i in range(n1)],
    "ACCOUNT_ID"       : [f"SGC{np.random.randint(10000,99999)}" for _ in range(n1)],
    "AMOUNT"           : np.random.lognormal(11, 2, n1).round(2),
    "CURRENCY"         : np.random.choice(["XOF","EUR","USD"], n1),
    "BOOKING_DATE"     : [hier] * n1,
    "TRANSACTION_TYPE" : np.random.choice(["VIR","PRE","CHQ"], n1),
})
# ✅ Aucune anomalie → toutes les dimensions OK → CSV exporté

# ── Extraction 2 : Comptes — Cohérence KO → pas d'export ─────────────────────
n2 = 180
df_comptes = pd.DataFrame({
    "ACCOUNT_ID"   : [f"SGC{np.random.randint(10000,99999)}" for _ in range(n2)],
    "CLIENT_NAME"  : [f"Client_{i}" for i in range(n2)],
    "ACCOUNT_TYPE" : np.random.choice(["CC","EP","CS","PRO"], n2),
    "OPEN_DATE"    : [hier - timedelta(days=np.random.randint(1,3650))] * n2,
    "BALANCE"      : np.random.lognormal(13, 2, n2).round(2),
    "STATUS"       : np.random.choice(["ACTIVE","INACTIVE"], n2),
})
df_comptes["BRANCH_CODE"] = np.nan  # ❌ colonne entièrement vide → Cohérence KO

# ── Extraction 3 : Mouvements — Validité KO → pas d'export ───────────────────
n3 = 400
df_mouvements = pd.DataFrame({
    "MVT_ID"       : range(n3),
    "ACCOUNT_ID"   : [f"SGC{np.random.randint(10000,99999)}" for _ in range(n3)],
    "DEBIT"        : np.random.lognormal(10, 2, n3).round(2),
    "CREDIT"       : np.random.lognormal(10, 2, n3).round(2),
    "MVT_DATE"     : [hier] * n3,
    "LABEL"        : [f"MVT_{i}" for i in range(n3)],
})
df_mouvements["MVT_ID"] = df_mouvements["MVT_ID"].astype(float)  # ❌ float au lieu d'int

print("✅ 3 DataFrames créés :")
print(f"   df_transactions : {len(df_transactions):,} lignes — aucune anomalie   → export attendu ✅")
print(f"   df_comptes      : {len(df_comptes):,} lignes — BRANCH_CODE vide    → pas d'export ⛔")
print(f"   df_mouvements   : {len(df_mouvements):,} lignes — MVT_ID type float → pas d'export ⛔")


## 6. Exemple avec 3 requêtes

> Chaque requête passe par `run_dq()` puis `log()`.
> Le CSV est exporté **uniquement** si toutes les dimensions sont OK.


In [ ]:
# ════════════════════════════════════════════════════════════
#  REQUÊTE 1 — Transactions journalières
#  Résultat attendu : ✅ VALIDE → CSV exporté
# ════════════════════════════════════════════════════════════

# En production :
# df_transactions = extraire("""
#     SELECT TRN_REF, ACCOUNT_ID, AMOUNT, CURRENCY,
#            BOOKING_DATE, TRANSACTION_TYPE
#     FROM SGCI_DB.TRANSACTIONS
#     WHERE BOOKING_DATE = CURRENT_DATE - 1
# """, "transactions_J1")

rapport_1 = run_dq(
    df             = df_transactions,
    nom            = "Transactions journalieres",
    min_lignes     = 50,
    max_lignes     = 1000,
    col_date       = "BOOKING_DATE",
    date_debut     = hier,
    date_fin       = hier,
    types_attendus = {
        "TRN_REF"          : str,
        "ACCOUNT_ID"       : str,
        "AMOUNT"           : float,
        "CURRENCY"         : str,
        "TRANSACTION_TYPE" : str,
    }
)
log("transactions_journalieres", rapport_1, "Quotidien", hier, hier)


In [ ]:
# ════════════════════════════════════════════════════════════
#  REQUÊTE 2 — Comptes clients actifs
#  Résultat attendu : ❌ NON VALIDE (Cohérence) → pas d'export
# ════════════════════════════════════════════════════════════

# En production :
# df_comptes = extraire("""
#     SELECT ACCOUNT_ID, CLIENT_NAME, ACCOUNT_TYPE,
#            OPEN_DATE, BALANCE, STATUS, BRANCH_CODE
#     FROM SGCI_DB.ACCOUNTS
#     WHERE STATUS = 'ACTIVE'
# """, "comptes_clients")

rapport_2 = run_dq(
    df             = df_comptes,
    nom            = "Comptes clients actifs",
    min_lignes     = 100,
    max_lignes     = 500,
    col_date       = "OPEN_DATE",
    date_debut     = date(2015, 1, 1),
    date_fin       = date.today(),
    types_attendus = {
        "ACCOUNT_ID"   : str,
        "BALANCE"      : float,
        "STATUS"       : str,
        "ACCOUNT_TYPE" : str,
    }
)
log("comptes_clients", rapport_2, "Quotidien", date(2015, 1, 1), date.today())


In [ ]:
# ════════════════════════════════════════════════════════════
#  REQUÊTE 3 — Mouvements comptables
#  Résultat attendu : ❌ NON VALIDE (Validité) → pas d'export
# ════════════════════════════════════════════════════════════

# En production :
# df_mouvements = extraire("""
#     SELECT MVT_ID, ACCOUNT_ID, DEBIT, CREDIT, MVT_DATE, LABEL
#     FROM SGCI_DB.MOUVEMENTS_COMPTABLES
#     WHERE MVT_DATE = CURRENT_DATE - 1
# """, "mouvements_comptables")

rapport_3 = run_dq(
    df             = df_mouvements,
    nom            = "Mouvements comptables",
    min_lignes     = 200,
    max_lignes     = 2000,
    col_date       = "MVT_DATE",
    date_debut     = hier,
    date_fin       = hier,
    types_attendus = {
        "MVT_ID"     : int,
        "ACCOUNT_ID" : str,
        "DEBIT"      : float,
        "CREDIT"     : float,
        "LABEL"      : str,
    }
)
log("mouvements_comptables", rapport_3, "Quotidien", hier, hier)


## 7. Bilan des exports

In [ ]:
# ── Résumé des 3 requêtes ─────────────────────────────────────────────────────
print("=" * 60)
print("  BILAN DES 3 REQUÊTES")
print("=" * 60)
for rapport in [rapport_1, rapport_2, rapport_3]:
    icone  = "✅" if rapport["valide"] else "❌"
    export = f"→ {os.path.basename(rapport['export_csv'])}" if rapport["export_csv"] else "→ pas d'export"
    print(f"  {icone} {rapport['nom']:<35} {export}")
print("=" * 60)

# ── Fichiers CSV exportés ─────────────────────────────────────────────────────
print(f"\n📁 Fichiers présents dans {EXPORT_DIR} :")
fichiers = list(Path(EXPORT_DIR).glob("*.csv"))
if fichiers:
    for f in fichiers:
        taille = round(f.stat().st_size / 1024, 1)
        print(f"   {f.name} ({taille} Ko)")
else:
    print("   (aucun fichier)")

# ── Journal ───────────────────────────────────────────────────────────────────
print(f"\n📋 Journal ({LOG_DIR}{LOG_FILE_NAME}) :")
df_journal = pd.read_csv(os.path.join(LOG_DIR, LOG_FILE_NAME), sep=";")
df_journal[["Extraction","Status","Row_Number","Export_CSV","Error_Reason"]]


---
## 📌 Template pour une nouvelle requête

Copie-colle ce bloc pour chaque nouvelle extraction :

```python
# 1. Extraire (en production)
df = extraire("""
    SELECT col1, col2, ma_date, ...
    FROM SGCI_DB.MA_TABLE
    WHERE ma_date = CURRENT_DATE - 1
""", "nom_extraction")

# 2. Contrôles DQ + export CSV si OK
rapport = run_dq(
    df             = df,
    nom            = "nom_extraction",
    min_lignes     = 100,           # ← à adapter
    max_lignes     = 5000,          # ← à adapter
    col_date       = "MA_DATE",     # ← colonne de date dans ta table
    date_debut     = hier,          # ← début de la période attendue
    date_fin       = hier,          # ← fin de la période attendue
    types_attendus = {              # ← types de tes colonnes clés
        "MA_CLE"      : str,
        "MON_MONTANT" : float,
    }
)

# 3. Journaliser
log("nom_extraction", rapport, "Quotidien", hier, hier)
```

Le CSV sera exporté dans `./EXPORTS/` uniquement si les 5 contrôles sont tous OK.
Le journal `./JOURNAL/journal_dq.csv` garde la trace de toutes les exécutions.

---
*SGCI Data Engineering · Framework DQ Simplifié v2.0 · Juin 2026*
